In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/A2AR/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="data/src/datasets/a2ar/data/A2AR.csv",
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
dataset.getDF().Y.sum()

3932

In [4]:
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=2, nBits=2048)],
    recalculate_features=True,
)

In [5]:
from qsprpred.data.descriptors.sets import RDKitDescs

rdkit_descs = RDKitDescs()

dataset.addDescriptors([rdkit_descs])

dataset.descriptorSets

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = dataset.X
X1, X2, y1, y2 = train_test_split(dataset.X, dataset.y, test_size=0.25, random_state=42)
X3 = dataset.X_ind
y3 = dataset.y_ind

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)

scaler = StandardScaler()
X1 = scaler.fit_transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)



In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,2248,2249,2250,2251,2252,2253,2254,2255,2256,2257
0,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,1.980336
1,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,2.943564,-0.181096,-0.225319,-0.258817
2,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.213839
3,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,1.331313
4,-0.05726,3.370999,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,4.222915,-2.480443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3533,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,1.656237,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.403167
3534,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,10.614778,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,0.983171,0.0,-0.281971,-0.181096,2.308674,1.364464
3535,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,1.656237,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.168575
3536,-0.05726,-0.296648,-0.105605,-0.070186,-0.075841,-0.053551,-0.067184,-0.603779,-0.132123,-0.150188,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.589209


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid

param_grid = {"n_estimators": [50, 100, 150],
                            "criterion": ["gini", "entropy"], 
                           "max_features": ["sqrt", "log2"],
                            "class_weight": ["balanced"],
                            "random_state": [42]
                           }
rf = RandomForestClassifier()
clf = GridSearchCV(rf, param_grid, scoring="f1")


In [18]:
clf.fit(X1, y1.values.ravel())

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'class_weight': ['balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [50, 100, 150], 'random_state': [42]},
             scoring='f1')

In [19]:
sorted(clf.cv_results_.keys())

['mean_fit_time',
 'mean_score_time',
 'mean_test_score',
 'param_class_weight',
 'param_criterion',
 'param_max_features',
 'param_n_estimators',
 'param_random_state',
 'params',
 'rank_test_score',
 'split0_test_score',
 'split1_test_score',
 'split2_test_score',
 'split3_test_score',
 'split4_test_score',
 'std_fit_time',
 'std_score_time',
 'std_test_score']

In [20]:
print("Nejlepší skóre na validační množině:", clf.best_score_)

Nejlepší skóre na validační množině: 0.9913217607259368


In [21]:
# Průměrné skóre na validačních množinách pro různé kombinace parametrů
validation_scores = clf.cv_results_['mean_test_score']
print("Průměrné skóre na validační množině pro různé parametry:")
print(validation_scores)

Průměrné skóre na validační množině pro různé parametry:
[0.98992658 0.98971359 0.98953587 0.98966984 0.99051789 0.99075757
 0.99053052 0.99051609 0.99053052 0.98968249 0.99132176 0.99053232]


In [22]:
best_params = clf.best_params_
print("Nejlepší hyperparametry:", best_params)

# Trénování modelu s těmito nejlepšími parametry:
best_rf = RandomForestClassifier(**best_params)  # Použití těchto parametrů
best_rf.fit(X1, y1)

# Aplikace na validační množinu:
y_pred = best_rf.predict(X2)  # X_val je tvoje validační množina

# Můžeš také vyhodnotit výsledky na validační množině:
from sklearn.metrics import classification_report
print(classification_report(y2, y_pred))

Nejlepší hyperparametry: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_features': 'log2', 'n_estimators': 100, 'random_state': 42}


/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

       False       0.67      0.29      0.40        35
        True       0.97      0.99      0.98       782

    accuracy                           0.96       817
   macro avg       0.82      0.64      0.69       817
weighted avg       0.96      0.96      0.96       817

